# LeetCode 2900 — Longest Unequal Adjacent Groups Subsequence I bằng Python

Link đề: https://leetcode.com/problems/longest-unequal-adjacent-groups-subsequence-i/ · Độ khó: **Easy** · Chủ đề: Array, Greedy, Dynamic Programming

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def getLongestSubsequence(self, words: List[str], groups: List[int]) -> List[str]:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code. Đề cho phép **nhiều đáp án đúng** (ví dụ 2 có hai đáp án), nên hãy đối chiếu lại với đề gốc về cách chấm.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có mảng chuỗi `words` và mảng `groups` (chỉ gồm `0` và `1`); chọn **dãy con dài nhất** của `words` sao cho hai phần tử được chọn **liền nhau** luôn thuộc **hai nhóm khác nhau**.
2. Hiểu khái niệm **dãy con (subsequence)** (giữ thứ tự, được bỏ phần tử) khác với **dãy con liên tiếp (subarray)**.
3. Đi từ cách **quy hoạch động kiểu LIS `O(n²)` có truy vết** → **tham lam so với nhóm của phần tử đã chọn cuối cùng** → **so với phần tử đứng ngay trước** → **`itertools.groupby` lấy phần tử đầu mỗi đoạn**.
4. Hiểu **vì sao tham lam đúng**: độ dài lớn nhất bằng **số đoạn** gồm các nhóm giống nhau liên tiếp, và lấy **một phần tử mỗi đoạn** là tối ưu.
5. Tránh các lỗi kinh điển: so sánh với phần tử đứng trước *thay vì* phần tử đã chọn gần nhất (ở bài khác), quên phần tử đầu tiên, trả về `groups` thay vì `words`, và nhầm "dãy con" với "đoạn liên tiếp".

## 2. Đề bài & ví dụ

Cho mảng chuỗi `words` gồm các chuỗi **khác nhau từng đôi một** và mảng nhị phân `groups` có cùng độ dài `n`; `words[i]` thuộc nhóm `groups[i]`.

Hãy chọn một **dãy con** các chỉ số `i0 < i1 < ... < i(k-1)` sao cho với mọi hai chỉ số được chọn **liền nhau** `ij` và `i(j+1)` ta có `groups[ij] != groups[i(j+1)]`. Trả về mảng `[words[i0], words[i1], ..., words[i(k-1)]]` có **độ dài lớn nhất** (nếu có nhiều đáp án, trả về một đáp án bất kỳ).

**Ví dụ 1**

- Input: `words = ["e", "a", "b"]`, `groups = [0, 0, 1]`
- Output: `["e", "b"]`
- Giải thích: chọn chỉ số `0` và `2` (nhóm `0` rồi `1`). `["a", "b"]` (chỉ số `1`, `2`) cũng đúng. Không thể chọn cả ba vì `groups[0] == groups[1]`.

**Ví dụ 2**

- Input: `words = ["a", "b", "c", "d"]`, `groups = [1, 0, 1, 1]`
- Output: `["a", "b", "c"]`
- Giải thích: chọn chỉ số `0, 1, 2` (nhóm `1, 0, 1`). `["a", "b", "d"]` cũng đúng. Độ dài lớn nhất là `3`.

**Ràng buộc:** `1 <= n == words.length == groups.length <= 100`; `1 <= words[i].length <= 10`; `groups[i]` là `0` hoặc `1`; các chuỗi trong `words` đều khác nhau.

In [1]:
from typing import List
from itertools import groupby
import random

examples = [
    (["e", "a", "b"], [0, 0, 1], ["e", "b"]),
    (["a", "b", "c", "d"], [1, 0, 1, 1], ["a", "b", "c"]),
]
for words, groups, expected in examples:
    print("words =", words, "groups =", groups, "-> kỳ vọng", expected)

words = ['e', 'a', 'b'] groups = [0, 0, 1] -> kỳ vọng ['e', 'b']
words = ['a', 'b', 'c', 'd'] groups = [1, 0, 1, 1] -> kỳ vọng ['a', 'b', 'c']


## 3. Kiến thức nền

### 3.1. Dãy con (subsequence) và đoạn liên tiếp (subarray)

- **Đoạn liên tiếp (subarray):** các phần tử **sát nhau**: `a[i:j]`.
- **Dãy con (subsequence):** chọn một số phần tử **giữ nguyên thứ tự**, có thể **bỏ qua** phần tử ở giữa. Ví dụ từ `[7, 8, 9]` có thể chọn `[7, 9]` (bỏ `8`).

Đề này tìm **dãy con**, nên được phép bỏ qua phần tử không phù hợp.

In [2]:
a = ["e", "a", "b"]
print(a[0:2])                       # đoạn liên tiếp: ['e', 'a']
picked = [a[i] for i in [0, 2]]     # dãy con theo danh sách chỉ số: bỏ qua chỉ số 1
print(picked)                        # ['e', 'b']

['e', 'a']
['e', 'b']


### 3.2. `enumerate` và `zip`

- `enumerate(a)` cho từng cặp `(chỉ_số, giá_trị)`.
- `zip(a, b)` ghép hai dãy theo từng vị trí. Với bài này, `zip(words, groups)` cho từng cặp `(từ, nhóm)`.

In [3]:
words = ["a", "b", "c", "d"]
groups = [1, 0, 1, 1]
for i, g in enumerate(groups):
    print("chỉ số", i, "nhóm", g)
for w, g in zip(words, groups):
    print(w, "thuộc nhóm", g)

chỉ số 0 nhóm 1
chỉ số 1 nhóm 0
chỉ số 2 nhóm 1
chỉ số 3 nhóm 1
a thuộc nhóm 1
b thuộc nhóm 0
c thuộc nhóm 1
d thuộc nhóm 1


### 3.3. Toán tử `!=` và so sánh hai phần tử kề nhau

Điều kiện của đề chỉ liên quan đến **nhóm của hai phần tử được chọn liền nhau**: `g1 != g2`. Với mảng, hai phần tử kề nhau là `groups[i - 1]` và `groups[i]` (`i >= 1`). Một danh sách "đúng" có `groups[i - 1] != groups[i]` ở **mọi** cặp kề.

In [4]:
groups = [1, 0, 1, 1]
for i in range(1, len(groups)):
    print(f"groups[{i-1}]={groups[i-1]} vs groups[{i}]={groups[i]} ->", "khác" if groups[i - 1] != groups[i] else "GIỐNG")

groups[0]=1 vs groups[1]=0 -> khác
groups[1]=0 vs groups[2]=1 -> khác
groups[2]=1 vs groups[3]=1 -> GIỐNG


### 3.4. Đoạn các nhóm giống nhau liên tiếp ("run")

Một **run** là đoạn liên tiếp các phần tử có **cùng giá trị** `groups`. Ví dụ `[1, 0, 1, 1]` có **3 run**: `[1]`, `[0]`, `[1, 1]`. Ví dụ `[0, 0, 1]` có **2 run**: `[0, 0]`, `[1]`. Số run bằng `1 + (số lần giá trị đổi giữa hai phần tử kề nhau)`.

In [5]:
def count_runs(groups):
    changes = sum(1 for i in range(1, len(groups)) if groups[i] != groups[i - 1])
    return 1 + changes


print(count_runs([1, 0, 1, 1]))      # 3
print(count_runs([0, 0, 1]))         # 2
print(count_runs([0, 0, 0]))         # 1

3
2
1


### 3.5. `itertools.groupby` trên cặp `(nhóm, từ)`

`groupby(dãy, key=hàm)` chia dãy thành **các đoạn liên tiếp có cùng khóa**; mỗi lần lặp trả `(khóa, nhóm_phần_tử)`. Để gom **run** theo `groups`, ta groupby trên `zip(groups, words)` với `key` là phần tử đầu của cặp (nhóm). Phần tử đầu tiên của mỗi nhóm chính là từ đầu tiên của run.

In [6]:
pairs = list(zip([1, 0, 1, 1], ["a", "b", "c", "d"]))
for key, run in groupby(pairs, key=lambda p: p[0]):
    run = list(run)
    print("nhóm", key, "->", run, "| từ đầu run:", run[0][1])

nhóm 1 -> [(1, 'a')] | từ đầu run: a
nhóm 0 -> [(0, 'b')] | từ đầu run: b
nhóm 1 -> [(1, 'c'), (1, 'd')] | từ đầu run: c


### 3.6. Đối chiếu một dãy con có hợp lệ không

Ta viết một hàm `is_valid_answer(words, groups, result)` để **kiểm tra** một đáp án: (1) mỗi từ của `result` thuộc `words` và thứ tự theo `words` tăng dần; (2) nhóm của hai từ liền nhau khác nhau. Vì các từ **khác nhau từng đôi một**, mỗi từ xác định đúng một chỉ số, nên ta tra chỉ số bằng `dict`.

In [7]:
def is_valid_answer(words, groups, result):
    index_of = {w: i for i, w in enumerate(words)}        # từ -> chỉ số (từ đều khác nhau)
    idx = [index_of[w] for w in result]                    # KeyError nếu từ không thuộc words
    if any(idx[j] >= idx[j + 1] for j in range(len(idx) - 1)):
        return False                                       # sai thứ tự (không phải dãy con)
    return all(groups[idx[j]] != groups[idx[j + 1]] for j in range(len(idx) - 1))


print(is_valid_answer(["e", "a", "b"], [0, 0, 1], ["e", "b"]))        # True
print(is_valid_answer(["e", "a", "b"], [0, 0, 1], ["e", "a", "b"]))   # False: e và a cùng nhóm 0
print(is_valid_answer(["e", "a", "b"], [0, 0, 1], ["b", "e"]))        # False: sai thứ tự

True
False
False


### 3.7. Quy hoạch động: "dãy tốt nhất kết thúc tại `i`"

`dp[i]` = độ dài dãy con hợp lệ **dài nhất kết thúc đúng tại chỉ số `i`**. Để tính `dp[i]`, nhìn các chỉ số trước `j < i` có nhóm **khác** `groups[i]` và lấy `dp[j]` lớn nhất, cộng `1`. Lưu `parent[i]` (chỉ số `j` được chọn) để **truy vết** dãy. Đây là cùng một ý tưởng với bài toán dãy con tăng dài nhất (LIS).

In [8]:
groups = [1, 0, 1, 1]
dp = [1] * len(groups)
for i in range(len(groups)):
    for j in range(i):
        if groups[j] != groups[i]:
            dp[i] = max(dp[i], dp[j] + 1)
print("dp =", dp)                     # [1, 2, 3, 2]
print("dài nhất =", max(dp))          # 3

dp = [1, 2, 3, 3]
dài nhất = 3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy xem `groups = [1, 0, 1, 1]` như một dãy bóng **đen/trắng**. Ta muốn xâu một chuỗi bóng **xen kẽ màu** (đen, trắng, đen, ...), được phép bỏ bớt bóng nhưng không được đảo thứ tự.

| chỉ số | nhóm | `dp[i]` (dài nhất kết thúc tại `i`) |
|---|---|---|
| 0 | 1 | 1 |
| 1 | 0 | 2 (nối sau chỉ số 0) |
| 2 | 1 | 3 (nối sau chỉ số 1) |
| 3 | 1 | 2 (chỉ nối được sau chỉ số 1, vì chỉ số 0 và 2 cùng nhóm 1) |

Độ dài lớn nhất là `3`. Nhận xét: khi gặp **hai bóng cùng màu liền nhau** (chỉ số 2 và 3 cùng nhóm 1), ta chỉ cần giữ **một** trong hai: giữ cái nào cũng được, vì mọi bóng sau đó chỉ "quan tâm" màu của bóng đã chọn. Vậy mỗi **run** (đoạn cùng màu liên tiếp) đóng góp **tối đa 1** phần tử.

## 5. Cách 1 — Brute Force / baseline: quy hoạch động `O(n²)` + truy vết

**Ý tưởng.** Tính `dp[i]` và `parent[i]` như mục 3.7. Chọn chỉ số `best` có `dp` lớn nhất, truy ngược `parent` để dựng chuỗi chỉ số rồi đảo lại và đổi sang `words`. Đây là cách "an toàn nhất" vì **không cần phát hiện ra tính chất đặc biệt** của dữ liệu nhị phân.

In [9]:
def longest_dp(words: List[str], groups: List[int]) -> List[str]:
    n = len(words)
    dp = [1] * n                      # dp[i]: dài nhất kết thúc tại i
    parent = [-1] * n                 # chỉ số đứng trước i trong dãy tối ưu
    for i in range(n):
        for j in range(i):            # thử mọi phần tử trước đó
            if groups[j] != groups[i] and dp[j] + 1 > dp[i]:
                dp[i] = dp[j] + 1
                parent[i] = j
    best = max(range(n), key=lambda i: dp[i])           # chỉ số kết thúc tốt nhất
    path = []
    while best != -1:                 # truy vết ngược
        path.append(words[best])
        best = parent[best]
    return path[::-1]


for words, groups, expected in examples:
    got = longest_dp(words, groups)
    print(got, "hợp lệ:", is_valid_answer(words, groups, got), "| độ dài", len(got), "(kỳ vọng", len(expected), ")")

['e', 'b'] hợp lệ: True | độ dài 2 (kỳ vọng 2 )
['a', 'b', 'c'] hợp lệ: True | độ dài 3 (kỳ vọng 3 )


**Độ phức tạp (Cách 1).** Hai vòng lồng nhau: **Time `O(n²)`**; mảng `dp`, `parent`, `path`: **Space `O(n)`**.

**Nút thắt.** `O(n²)` là thừa. Vì `groups` chỉ có **hai giá trị**, "khác nhóm" nghĩa là "nhóm đối ngược", và ta không cần thử mọi `j` mà chỉ cần nhớ **nhóm của phần tử vừa chọn**.

## 6. Ý tưởng thuật toán / bước đột phá

**Tham lam.** Duyệt từ trái sang phải, giữ **nhóm của phần tử được chọn gần nhất** (`last`). Gặp phần tử có nhóm **khác `last`** thì **chọn nó ngay** và cập nhật `last`. Phần tử đầu tiên luôn được chọn.

Vì sao đúng (phác họa):

- Một dãy con hợp lệ không thể lấy hai phần tử **cùng một run** (hai phần tử cùng run mà muốn kề nhau trong dãy con thì nhóm bằng nhau, vi phạm; còn nếu giữa chúng có phần tử khác trong dãy con thì phần tử đó phải thuộc run khác, mâu thuẫn vì run là đoạn liên tiếp). Nên số phần tử tối đa **bằng số run**.
- Cách chọn "phần tử đầu mỗi run" đạt đúng số đó.

Vậy đáp án dài `số run`, và tham lam (`O(n)`) dựng được nó.

## 7. Các cách tiếp theo

### Cách 2 — Tham lam với `last` (nhóm của phần tử đã chọn cuối cùng)

**Giải thích.** Biến `last` ban đầu là `-1` (không trùng `0` hay `1`) để phần tử đầu tiên luôn được chọn. Mỗi lần `groups[i] != last` thì thêm `words[i]` và `last = groups[i]`.

In [10]:
def longest_greedy_last(words: List[str], groups: List[int]) -> List[str]:
    result = []
    last = -1                          # nhóm của phần tử đã chọn gần nhất; -1 = chưa chọn gì
    for w, g in zip(words, groups):
        if g != last:                  # khác nhóm với phần tử vừa chọn -> chọn
            result.append(w)
            last = g
    return result


for words, groups, expected in examples:
    print(longest_greedy_last(words, groups), "(kỳ vọng", expected, ")")

['e', 'b'] (kỳ vọng ['e', 'b'] )
['a', 'b', 'c'] (kỳ vọng ['a', 'b', 'c'] )


In [11]:
# Code nộp LeetCode cho Cách 2
from typing import List

class Solution:
    def getLongestSubsequence(self, words: List[str], groups: List[int]) -> List[str]:
        result = []
        last = -1
        for w, g in zip(words, groups):
            if g != last:
                result.append(w)
                last = g
        return result

**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(n)`** cho mảng kết quả (hoặc `O(1)` phụ không tính đáp án).

### Cách 3 — So sánh với phần tử đứng ngay trước (`groups[i] != groups[i - 1]`)

**Giải thích.** Chọn `words[0]`, rồi với `i >= 1` chọn `words[i]` nếu `groups[i] != groups[i - 1]`. Cho đúng cùng kết quả Cách 2 vì "phần tử đã chọn gần nhất" luôn cùng nhóm với phần tử đứng liền trước (cùng run).

In [12]:
def longest_greedy_prev(words: List[str], groups: List[int]) -> List[str]:
    result = [words[0]]                                  # phần tử đầu luôn được chọn
    for i in range(1, len(words)):
        if groups[i] != groups[i - 1]:                   # nhóm đổi so với phần tử kề trước
            result.append(words[i])
    return result


for words, groups, expected in examples:
    print(longest_greedy_prev(words, groups), "(kỳ vọng", expected, ")")

['e', 'b'] (kỳ vọng ['e', 'b'] )
['a', 'b', 'c'] (kỳ vọng ['a', 'b', 'c'] )


In [13]:
# Code nộp LeetCode cho Cách 3
from typing import List

class Solution:
    def getLongestSubsequence(self, words: List[str], groups: List[int]) -> List[str]:
        return [words[0]] + [words[i] for i in range(1, len(words)) if groups[i] != groups[i - 1]]

**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(n)`**. Cần `n >= 1` (đề đảm bảo `n >= 1`).

### Cách 4 — `itertools.groupby`: lấy từ đầu của mỗi run

**Giải thích.** Gom `zip(groups, words)` theo nhóm liên tiếp; mỗi run lấy phần tử đầu tiên. Cách này **nói thẳng ý tưởng**: "mỗi run đóng góp một từ".

In [14]:
def longest_groupby(words: List[str], groups: List[int]) -> List[str]:
    result = []
    for _, run in groupby(zip(groups, words), key=lambda p: p[0]):
        first_pair = next(run)               # cặp (nhóm, từ) đầu tiên của run
        result.append(first_pair[1])
    return result


for words, groups, expected in examples:
    print(longest_groupby(words, groups), "(kỳ vọng", expected, ")")

['e', 'b'] (kỳ vọng ['e', 'b'] )
['a', 'b', 'c'] (kỳ vọng ['a', 'b', 'c'] )


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(n)`**.

> Ba cách tham lam (2, 3, 4) cho đúng **cùng một đáp án** (lấy từ đầu mỗi run). Cách 1 (DP) có thể ra đáp án **khác** nhưng cùng độ dài (đều hợp lệ). Cách 2 là cách tổng quát nhất; Cách 3 ngắn nhất.

## 8. TRACE từng bước

Ta theo dõi Cách 2 (tham lam) trên ví dụ dài hơn `groups = [1, 1, 0, 0, 0, 1, 0, 0]` và Cách 1 (DP) trên ví dụ của đề.

In [15]:
def trace_greedy(words, groups):
    result, last = [], -1
    print("words =", words)
    print("groups =", groups)
    for i, (w, g) in enumerate(zip(words, groups)):
        if g != last:
            result.append(w)
            print(f"i={i}: w={w!r}, g={g}, last={last:>2} -> KHÁC, chọn; result={result}")
            last = g
        else:
            print(f"i={i}: w={w!r}, g={g}, last={last:>2} -> GIỐNG, bỏ qua")
    print("=> độ dài", len(result), "=", count_runs(groups), "(số run)")
    return result


out = trace_greedy(list("abcdefgh"), [1, 1, 0, 0, 0, 1, 0, 0])
assert out == ["a", "c", "f", "g"]

words = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']
groups = [1, 1, 0, 0, 0, 1, 0, 0]
i=0: w='a', g=1, last=-1 -> KHÁC, chọn; result=['a']
i=1: w='b', g=1, last= 1 -> GIỐNG, bỏ qua
i=2: w='c', g=0, last= 1 -> KHÁC, chọn; result=['a', 'c']
i=3: w='d', g=0, last= 0 -> GIỐNG, bỏ qua
i=4: w='e', g=0, last= 0 -> GIỐNG, bỏ qua
i=5: w='f', g=1, last= 0 -> KHÁC, chọn; result=['a', 'c', 'f']
i=6: w='g', g=0, last= 1 -> KHÁC, chọn; result=['a', 'c', 'f', 'g']
i=7: w='h', g=0, last= 0 -> GIỐNG, bỏ qua
=> độ dài 4 = 4 (số run)


In [16]:
def trace_dp(words, groups):
    n = len(words)
    dp, parent = [1] * n, [-1] * n
    for i in range(n):
        for j in range(i):
            if groups[j] != groups[i] and dp[j] + 1 > dp[i]:
                dp[i], parent[i] = dp[j] + 1, j
        print(f"i={i}: groups[i]={groups[i]} dp[i]={dp[i]} parent[i]={parent[i]}")
    best = max(range(n), key=lambda i: dp[i])
    path = []
    while best != -1:
        path.append(best)
        best = parent[best]
    path.reverse()
    print("chuỗi chỉ số:", path, "->", [words[i] for i in path])
    return [words[i] for i in path]


out = trace_dp(["a", "b", "c", "d"], [1, 0, 1, 1])
assert is_valid_answer(["a", "b", "c", "d"], [1, 0, 1, 1], out) and len(out) == 3

i=0: groups[i]=1 dp[i]=1 parent[i]=-1
i=1: groups[i]=0 dp[i]=2 parent[i]=0
i=2: groups[i]=1 dp[i]=3 parent[i]=1
i=3: groups[i]=1 dp[i]=3 parent[i]=1
chuỗi chỉ số: [0, 1, 2] -> ['a', 'b', 'c']


Bảng tóm tắt (tham lam, `groups = [1, 1, 0, 0, 0, 1, 0, 0]`):

| i | nhóm | `last` trước | hành động | `result` |
|---|---|---|---|---|
| 0 | 1 | -1 | chọn | `[a]` |
| 1 | 1 | 1 | bỏ | `[a]` |
| 2 | 0 | 1 | chọn | `[a, c]` |
| 3 | 0 | 0 | bỏ | `[a, c]` |
| 4 | 0 | 0 | bỏ | `[a, c]` |
| 5 | 1 | 0 | chọn | `[a, c, f]` |
| 6 | 0 | 1 | chọn | `[a, c, f, g]` |
| 7 | 0 | 0 | bỏ | `[a, c, f, g]` |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (tham lam với `last`). Đúng class `Solution`, đúng method `getLongestSubsequence(self, words: List[str], groups: List[int]) -> List[str]`.

In [17]:
from typing import List

class Solution:
    def getLongestSubsequence(self, words: List[str], groups: List[int]) -> List[str]:
        result: List[str] = []
        last = -1                                # nhóm của từ được chọn gần nhất
        for w, g in zip(words, groups):
            if g != last:
                result.append(w)
                last = g
        return result


sol = Solution()
for words, groups, expected in examples:
    print(sol.getLongestSubsequence(words, groups))

['e', 'b']
['a', 'b', 'c']


## 10. Vì sao thuật toán đúng

- **Hợp lệ:** mỗi lần chọn một phần tử, ta so với `last` (nhóm của phần tử đã chọn ngay trước) và chỉ chọn khi **khác**; nên hai phần tử được chọn liền nhau luôn khác nhóm. Thứ tự giữ nguyên vì ta duyệt trái sang phải.
- **Cận trên:** chia `groups` thành các **run** (đoạn cùng giá trị liên tiếp). Giả sử dãy con hợp lệ có hai phần tử `i < j` cùng một run. Mọi phần tử giữa `i` và `j` cũng thuộc run này (run liên tục) và cùng giá trị nhóm. Trong dãy con, các phần tử từ `i` đến `j` đều thuộc run đó nên cùng nhóm với nhau; nhưng hai phần tử **liền nhau trong dãy con** phải khác nhóm, mâu thuẫn. Vậy mỗi run chứa **tối đa 1** phần tử được chọn, độ dài tối đa là số run `R`.
- **Đạt cận trên:** thuật toán chọn đúng **phần tử đầu mỗi run**: ở phần tử đầu của run mới `groups[i] != groups[i - 1] = last` nên được chọn; các phần tử còn lại của run bằng `last` nên bị bỏ. Do đó chọn đúng `R` phần tử.
- **Kết luận:** độ dài `R` vừa là cận trên vừa đạt được nên tối ưu.
- **Tính chất kiểm được:** `len(result) == số run`; kết quả qua `is_valid_answer`.

## 11. Phân tích Time / Space Complexity

Gọi `n` là độ dài `words`.

- **Cách 1 (DP):** Time `O(n²)`, Space `O(n)`.
- **Cách 2, 3, 4 (tham lam):** Time **`O(n)`** (mỗi phần tử đúng một lần), Space **`O(n)`** cho mảng kết quả (phụ thêm `O(1)`).
- Không thể tốt hơn `O(n)` vì cần đọc mọi nhóm để biết chỗ đổi.

## 12. Bộ test

Vì đề cho phép nhiều đáp án, ta so sánh bằng **độ dài tối ưu** (tính bằng DP độc lập) và **tính hợp lệ** (`is_valid_answer`); riêng ba cách tham lam còn phải khớp nhau từng phần tử.

In [18]:
sol = Solution()
greedy_impls = {
    "greedy_last": longest_greedy_last,
    "greedy_prev": longest_greedy_prev,
    "groupby": longest_groupby,
    "submit": sol.getLongestSubsequence,
}
all_impls = {"dp": longest_dp, **greedy_impls}


def optimal_length(groups):
    """Độ dài tối ưu tính bằng DP (độc lập với các cách tham lam)."""
    dp = [1] * len(groups)
    for i in range(len(groups)):
        for j in range(i):
            if groups[j] != groups[i]:
                dp[i] = max(dp[i], dp[j] + 1)
    return max(dp)


for words, groups, expected in examples:
    for key, fn in all_impls.items():
        got = fn(words, groups)
        assert is_valid_answer(words, groups, got), (key, words, groups, got)
        assert len(got) == len(expected) == optimal_length(groups), (key, got)
    # ví dụ có đáp án duy nhất theo cách tham lam
    assert sol.getLongestSubsequence(words, groups) == expected
print("PASS ví dụ của đề (mọi cách hợp lệ, đủ độ dài; bản nộp khớp đáp án mẫu)")

PASS ví dụ của đề (mọi cách hợp lệ, đủ độ dài; bản nộp khớp đáp án mẫu)


### 12.1. Các trường hợp biên

In [19]:
def check_all(words, groups, label):
    k = optimal_length(groups)
    for key, fn in all_impls.items():
        got = fn(words, groups)
        assert is_valid_answer(words, groups, got), (label, key, got)
        assert len(got) == k, (label, key, len(got), k)
    greedy_answers = {tuple(fn(words, groups)) for fn in greedy_impls.values()}
    assert len(greedy_answers) == 1, (label, greedy_answers)
    print("PASS", label, "-> độ dài", k, "|", sol.getLongestSubsequence(words, groups))


check_all(["a"], [0], "một phần tử")
check_all(["a", "b"], [0, 0], "hai phần tử cùng nhóm -> chỉ chọn 1")
check_all(["a", "b"], [0, 1], "hai phần tử khác nhóm -> chọn cả hai")
check_all(["a", "b", "c", "d", "e"], [0, 0, 0, 0, 0], "toàn cùng nhóm")
check_all(["a", "b", "c", "d", "e"], [0, 1, 0, 1, 0], "xen kẽ hoàn hảo -> chọn tất cả")
check_all(["a", "b", "c", "d", "e", "f"], [1, 1, 1, 0, 0, 0], "hai run lớn")
check_all(["a", "b", "c", "d", "e", "f"], [0, 1, 1, 0, 0, 1], "run độ dài khác nhau")

PASS một phần tử -> độ dài 1 | ['a']
PASS hai phần tử cùng nhóm -> chỉ chọn 1 -> độ dài 1 | ['a']
PASS hai phần tử khác nhóm -> chọn cả hai -> độ dài 2 | ['a', 'b']
PASS toàn cùng nhóm -> độ dài 1 | ['a']
PASS xen kẽ hoàn hảo -> chọn tất cả -> độ dài 5 | ['a', 'b', 'c', 'd', 'e']
PASS hai run lớn -> độ dài 2 | ['a', 'd']
PASS run độ dài khác nhau -> độ dài 4 | ['a', 'b', 'd', 'f']


### 12.2. Tính chất và phản ví dụ

In [20]:
# Tính chất: độ dài đáp án = số run
random.seed(5)
for _ in range(200):
    n = random.randint(1, 30)
    groups = [random.randint(0, 1) for _ in range(n)]
    words = [f"w{i}" for i in range(n)]
    assert len(sol.getLongestSubsequence(words, groups)) == count_runs(groups)
print("PASS len(đáp án) = số run")

# Tính chất: mỗi từ chọn là từ đầu của một run (chỉ số 0 hoặc nhóm đổi)
words, groups = list("abcdefgh"), [1, 1, 0, 0, 0, 1, 0, 0]
res = sol.getLongestSubsequence(words, groups)
starts = [words[i] for i in range(len(words)) if i == 0 or groups[i] != groups[i - 1]]
assert res == starts
print("PASS đáp án = các từ đầu của mỗi run:", res)

# Đầu vào không bị sửa
w0, g0 = list(words), list(groups)
for key, fn in all_impls.items():
    fn(words, groups)
assert words == w0 and groups == g0
print("PASS đầu vào không bị sửa")

# Phản ví dụ 1: trả về groups thay vì words
def sai_tra_groups(words, groups):
    out, last = [], -1
    for w, g in zip(words, groups):
        if g != last:
            out.append(g)
            last = g
    return out

assert sai_tra_groups(["a", "b", "c", "d"], [1, 0, 1, 1]) == [1, 0, 1]
print("PASS phản ví dụ: trả về", sai_tra_groups(["a", "b", "c", "d"], [1, 0, 1, 1]), "(là nhóm) thay vì từ")

# Phản ví dụ 2: lấy các phần tử bằng nhóm với phần tử đầu -> không hợp lệ
sai = [w for w, g in zip(["a", "b", "c", "d"], [1, 0, 1, 1]) if g == 1]
assert not is_valid_answer(["a", "b", "c", "d"], [1, 0, 1, 1], sai)
print("PASS phản ví dụ:", sai, "không hợp lệ (c và d cùng nhóm)")

PASS len(đáp án) = số run
PASS đáp án = các từ đầu của mỗi run: ['a', 'c', 'f', 'g']
PASS đầu vào không bị sửa
PASS phản ví dụ: trả về [1, 0, 1] (là nhóm) thay vì từ
PASS phản ví dụ: ['a', 'c', 'd'] không hợp lệ (c và d cùng nhóm)


### 12.3. Đối chiếu ngẫu nhiên (có seed)

So sánh độ dài tối ưu của DP với từng cách tham lam, kiểm tra tính hợp lệ, và ba cách tham lam khớp nhau.

In [21]:
random.seed(2900)
for trial in range(3000):
    n = random.randint(1, 100)
    p = random.choice([0.1, 0.5, 0.9])
    groups = [1 if random.random() < p else 0 for _ in range(n)]
    words = [f"w{i}" for i in range(n)]
    k = optimal_length(groups)
    answers = set()
    for key, fn in all_impls.items():
        got = fn(words, groups)
        assert is_valid_answer(words, groups, got), (key, groups)
        assert len(got) == k, (key, len(got), k)
        if key != "dp":
            answers.add(tuple(got))
    assert len(answers) == 1
print("PASS 3000 bộ ngẫu nhiên - độ dài tối ưu, hợp lệ, ba cách tham lam khớp nhau")

PASS 3000 bộ ngẫu nhiên - độ dài tối ưu, hợp lệ, ba cách tham lam khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên phần tử đầu tiên

In [22]:
def sai_quen_dau(words, groups):
    return [words[i] for i in range(1, len(words)) if groups[i] != groups[i - 1]]


print("sai  :", sai_quen_dau(["a", "b", "c", "d"], [1, 0, 1, 1]))               # thiếu 'a'
print("đúng :", Solution().getLongestSubsequence(["a", "b", "c", "d"], [1, 0, 1, 1]))

sai  : ['b', 'c']
đúng : ['a', 'b', 'c']


**Vì sao sai:** mỗi **run** đóng góp một phần tử, và **run đầu tiên** không có phần tử "đứng trước" để so sánh nên bị bỏ qua. Phải luôn thêm `words[0]`, hoặc dùng `last = -1` để phần tử đầu luôn được chọn.

### Lỗi 2: Khởi tạo `last = 0` (trùng một nhóm hợp lệ)

In [23]:
def sai_last_bang_0(words, groups):
    result, last = [], 0                     # nhóm 0 cũng là nhóm thật!
    for w, g in zip(words, groups):
        if g != last:
            result.append(w)
            last = g
    return result


print("sai  :", sai_last_bang_0(["a", "b"], [0, 1]))      # ['b']: thiếu 'a' vì g(0) == last(0)
print("đúng :", Solution().getLongestSubsequence(["a", "b"], [0, 1]))

sai  : ['b']
đúng : ['a', 'b']


**Vì sao sai:** nếu `groups[0] == 0` mà `last` khởi tạo bằng `0` thì phần tử đầu bị coi "cùng nhóm với phần tử trước nó" và bị bỏ. Dùng giá trị **không thuộc `{0, 1}`** (ví dụ `-1`) hoặc `None` để biểu thị "chưa chọn gì".

### Lỗi 3: Nhầm "dãy con" với "đoạn liên tiếp"

In [24]:
def sai_doan_lien_tiep(words, groups):
    # tìm đoạn liên tiếp dài nhất mà hai phần tử kề nhau luôn khác nhóm
    best, cur = [words[0]], [words[0]]
    for i in range(1, len(words)):
        if groups[i] != groups[i - 1]:
            cur.append(words[i])
        else:
            cur = [words[i]]
        if len(cur) > len(best):
            best = list(cur)
    return best


words3, groups3 = ["x", "a", "b", "c", "y", "z"], [0, 0, 1, 0, 0, 1]
print("sai  :", sai_doan_lien_tiep(words3, groups3))                   # ['a', 'b', 'c']: dài 3
print("đúng :", Solution().getLongestSubsequence(words3, groups3))     # ['x', 'b', 'c', 'z']: dài 4
print("tối ưu:", optimal_length(groups3))

sai  : ['a', 'b', 'c']
đúng : ['x', 'b', 'c', 'z']
tối ưu: 4


**Vì sao sai:** đoạn liên tiếp **không được bỏ** phần tử ở giữa, nên khi gặp hai phần tử cùng nhóm nó phải bắt đầu lại và có thể bỏ lỡ dãy dài hơn. Đề cho phép **bỏ phần tử** (dãy con), như ở ví dụ trên: đáp án đúng dài `4` (`x, b, c, z`), còn đoạn liên tiếp chỉ tìm được dài `3` (`a, b, c`).

### Lỗi 4: Dùng `groups[i] != groups[i - 1]` với `i = 0` (chỉ số âm)

In [25]:
def sai_chi_so_am(words, groups):
    return [words[i] for i in range(len(words)) if groups[i] != groups[i - 1]]   # i = 0 so với groups[-1]!


print("sai  :", sai_chi_so_am(["a", "b", "c"], [1, 0, 1]))       # ['b', 'c']: 'a' bị so với phần tử CUỐI (cùng nhóm 1) nên bị bỏ oan
print("đúng :", Solution().getLongestSubsequence(["a", "b", "c"], [1, 0, 1]))

sai  : ['b', 'c']
đúng : ['a', 'b', 'c']


**Vì sao sai:** `groups[-1]` là **phần tử cuối**, không phải "không có gì". Ở `i = 0` ta so sánh nhầm với phần tử cuối và nếu hai nhóm bằng nhau thì phần tử đầu bị bỏ oan. Phải xử lý `i = 0` riêng (như Cách 3) hoặc dùng `last`.

### Lỗi 5: Tin rằng phải dùng DP vì "bài dãy con"

In [26]:
n = 100
words = [f"w{i}" for i in range(n)]
groups = [random.randint(0, 1) for _ in range(n)]
a = longest_dp(words, groups)
b = Solution().getLongestSubsequence(words, groups)
print("DP và tham lam cho cùng độ dài:", len(a), len(b))
print("số phép so sánh nhóm: DP =", n * (n - 1) // 2, "| tham lam =", n)
assert len(a) == len(b)

DP và tham lam cho cùng độ dài: 50 50
số phép so sánh nhóm: DP = 4950 | tham lam = 100


**Vì sao dễ sa vào:** nhiều bài "dãy con dài nhất" cần quy hoạch động. Ở đây dữ liệu chỉ có **hai giá trị**, tạo ra cấu trúc "run" nên tham lam đúng và nhanh hơn. Biết cả hai cách giúp bạn kiểm chứng: DP là "chuẩn đáng tin" để đối chiếu, tham lam là cách nộp. (Bài II cùng series yêu cầu thêm điều kiện về khoảng cách Hamming nên cần DP thực sự.)

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "dãy con dài nhất mà hai phần tử kề nhau phải khác nhau (nhị phân)" → nghĩ tới **đếm số run** và **lấy đầu mỗi run**.
- Công thức nhớ: `[words[0]] + [words[i] for i in range(1, n) if groups[i] != groups[i - 1]]`.
- Khi không chắc tham lam: viết **DP `O(n²)`** làm bản đối chiếu và so với tham lam bằng dữ liệu ngẫu nhiên.
- Giá trị khởi tạo `last` phải là giá trị **ngoài miền** (`-1`/`None`).
- Đề cho phép nhiều đáp án: kiểm tra bằng **hàm hợp lệ + độ dài tối ưu**, không so sánh bằng `==` với một đáp án mẫu.
- Hai chuỗi khác nhau từng đôi một cho phép tra chỉ số bằng `dict`.

## 15. Biến thể của bài

1. **Bài II** (cùng đợt): `groups` là số nguyên bất kỳ và thêm điều kiện **khoảng cách Hamming = 1** giữa hai từ liền nhau trong dãy con; cần DP `O(n²·L)`.
2. Chỉ **đếm độ dài** lớn nhất (không dựng dãy): `count_runs(groups)`.
3. Chọn **phần tử cuối** mỗi run thay vì phần tử đầu (đáp án khác nhưng vẫn hợp lệ).
4. `groups` có **nhiều hơn hai giá trị**: vẫn là "bất kỳ hai phần tử liền nhau khác nhóm" nhưng tham lam khi đó **vẫn** chọn đầu mỗi run (chỉ cần khác nhóm phần tử trước).
5. Dãy con có **tổng trọng số lớn nhất** thay vì dài nhất: cần DP có trọng số.
6. Trả về **mọi** đáp án dài nhất (đếm số cách): kết hợp nhân số lựa chọn trong từng run (`độ dài run` mỗi run).

Ta thử biến thể 2, 3 và 6:

In [27]:
words = list("abcdefgh")
groups = [1, 1, 0, 0, 0, 1, 0, 0]
print("2. độ dài tối ưu :", count_runs(groups))

last_of_run = [words[i] for i in range(len(words)) if i == len(words) - 1 or groups[i] != groups[i + 1]]
print("3. phần tử cuối mỗi run:", last_of_run, "| hợp lệ:", is_valid_answer(words, groups, last_of_run))

# 6. Số đáp án dài nhất = tích độ dài các run
run_lengths = [len(list(run)) for _, run in groupby(groups)]
ways = 1
for r in run_lengths:
    ways *= r
print("6. độ dài các run:", run_lengths, "-> số đáp án dài nhất:", ways)

# kiểm chứng bằng cách liệt kê (n nhỏ)
from itertools import combinations
k = count_runs(groups)
count = sum(1 for idx in combinations(range(len(words)), k) if all(groups[idx[j]] != groups[idx[j + 1]] for j in range(k - 1)))
print("   kiểm chứng bằng liệt kê:", count)
assert count == ways

2. độ dài tối ưu : 4
3. phần tử cuối mỗi run: ['b', 'e', 'f', 'h'] | hợp lệ: True
6. độ dài các run: [2, 3, 1, 2] -> số đáp án dài nhất: 12
   kiểm chứng bằng liệt kê: 12


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. DP `O(n²)` + truy vết | dài nhất kết thúc tại `i` | `O(n²)` | `O(n)` | bản đối chiếu; tổng quát (cho các biến thể có điều kiện phức tạp) |
| 2. Tham lam với `last` | chọn khi khác nhóm phần tử vừa chọn | `O(n)` | `O(n)` | **cách nộp bài**, tổng quát nhất trong nhóm tham lam |
| 3. So với `groups[i-1]` | chọn đầu mỗi run | `O(n)` | `O(n)` | ngắn nhất (một dòng) |
| 4. `groupby` | lấy phần tử đầu mỗi run | `O(n)` | `O(n)` | khi muốn nói thẳng "mỗi run một phần tử" |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `count_runs([0, 0, 1, 1, 0])`, `count_runs([1])`, `Solution().getLongestSubsequence(["x", "y", "z"], [0, 0, 0])`, `Solution().getLongestSubsequence(["x", "y", "z"], [0, 1, 0])`, `[w for w, g in zip("abc", [1, 0, 1]) if g == 1]`.

**Bài 2 (tính tay).** Với `words = [p, q, r, s, t, u]`, `groups = [0, 0, 1, 1, 1, 0]`: viết các run, độ dài lớn nhất, và đáp án của tham lam.

**Bài 3 (điền TODO).** Hoàn thành `longest_todo(words, groups)` theo Cách 2.

In [28]:
def longest_todo(words, groups):
    result = []
    last = -1
    for w, g in zip(words, groups):
        # TODO 1: nếu g khác last thì thêm w vào result
        # TODO 2: và cập nhật last = g
        pass
    return result

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `longest_dp2(words, groups)` bằng quy hoạch động có truy vết (Cách 1) mà không nhìn code mẫu. Kiểm tra bằng `is_valid_answer` và độ dài tối ưu.

**Bài 5 (biến thể / mini challenge).** (a) Viết `longest_indices(groups)` trả về **danh sách chỉ số** được chọn (không cần `words`). (b) Viết `count_longest(groups)` trả về **số lượng dãy con dài nhất** (số cách chọn một phần tử cho mỗi run), mod `10**9 + 7`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `3`, `1`, `['x']`, `['x', 'y', 'z']`, `['a', 'c']`. Bài 2: run là đoạn liên tiếp cùng nhóm.
- **Mức 2 (vừa):** Bài 2: run `[p, q]` (nhóm 0), `[r, s, t]` (nhóm 1), `[u]` (nhóm 0): 3 run; độ dài tối đa 3; tham lam chọn `p, r, u`. Bài 3: hai TODO đã ghi trong comment. Bài 4: `dp = [1] * n`, `parent = [-1] * n`, cập nhật khi `groups[j] != groups[i]`, rồi truy vết từ chỉ số có `dp` lớn nhất.
- **Mức 3 (sâu):** Bài 5(a): giống Cách 2 nhưng thêm `i` thay vì `w`. Bài 5(b): `ways = 1`; với mỗi run độ dài `r`: `ways = ways * r % MOD`; dùng `groupby(groups)` để lấy độ dài run.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [29]:
# Bài 1
assert count_runs([0, 0, 1, 1, 0]) == 3 and count_runs([1]) == 1
assert Solution().getLongestSubsequence(["x", "y", "z"], [0, 0, 0]) == ["x"]
assert Solution().getLongestSubsequence(["x", "y", "z"], [0, 1, 0]) == ["x", "y", "z"]
assert [w for w, g in zip("abc", [1, 0, 1]) if g == 1] == ["a", "c"]

# Bài 2
w2, g2 = list("pqrstu"), [0, 0, 1, 1, 1, 0]
assert count_runs(g2) == 3
assert Solution().getLongestSubsequence(w2, g2) == ["p", "r", "u"]

# Bài 3
def longest_todo_ans(words, groups):
    result = []
    last = -1
    for w, g in zip(words, groups):
        if g != last:
            result.append(w)
            last = g
    return result

assert longest_todo_ans(w2, g2) == ["p", "r", "u"]
for words, groups, expected in examples:
    assert longest_todo_ans(words, groups) == expected

# Bài 4
def longest_dp2(words, groups):
    n = len(words)
    dp, parent = [1] * n, [-1] * n
    for i in range(n):
        for j in range(i):
            if groups[j] != groups[i] and dp[j] + 1 > dp[i]:
                dp[i], parent[i] = dp[j] + 1, j
    cur = max(range(n), key=lambda i: dp[i])
    path = []
    while cur != -1:
        path.append(words[cur])
        cur = parent[cur]
    return path[::-1]

for words, groups, expected in examples:
    got = longest_dp2(words, groups)
    assert is_valid_answer(words, groups, got) and len(got) == optimal_length(groups)

# Bài 5(a)
def longest_indices(groups):
    idx, last = [], -1
    for i, g in enumerate(groups):
        if g != last:
            idx.append(i)
            last = g
    return idx

assert longest_indices([1, 1, 0, 0, 0, 1, 0, 0]) == [0, 2, 5, 6]

# Bài 5(b)
def count_longest(groups):
    MOD = 10**9 + 7
    ways = 1
    for _, run in groupby(groups):
        ways = ways * len(list(run)) % MOD
    return ways

assert count_longest([1, 1, 0, 0, 0, 1, 0, 0]) == 2 * 3 * 1 * 2
assert count_longest([0]) == 1
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đầu mỗi run khác nhóm)

```python
from typing import List

class Solution:
    def getLongestSubsequence(self, words: List[str], groups: List[int]) -> List[str]:
        result, last = [], -1
        for w, g in zip(words, groups):
            if g != last:
                result.append(w)
                last = g
        return result
```

**Checklist**

- [ ] `last` khởi tạo bằng giá trị **ngoài miền** (`-1`), không phải `0`?
- [ ] Phần tử đầu tiên luôn được chọn?
- [ ] Trả về mảng **`words`** được chọn (không phải `groups`, không phải chỉ số)?
- [ ] So sánh với nhóm của phần tử **đã chọn gần nhất** (hoặc phần tử kề trước ở bài nhị phân), không so với `groups[-1]` khi `i = 0`?
- [ ] Hiểu "dãy con" được bỏ phần tử, khác "đoạn liên tiếp"?
- [ ] Đã thử: một phần tử, toàn cùng nhóm, xen kẽ hoàn hảo?
- [ ] Skeleton đúng `class Solution` / `def getLongestSubsequence(self, words: List[str], groups: List[int]) -> List[str]`?